Objective: Detect media bias using Logistic Regression.

Goal: To build a binary text classifier that identifies subjective media bias. This tool should help public by highlighting biased language in news media.

Binary prediction: Biased(1) vs. Neutral(0)

Dataset:  newsmediabias/debiased_dataset (from HuggingFace)

In [ ]:
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# Load the dataset
print("Loading dataset...")
dataset = load_dataset("newsmediabias/debiased_dataset", split="train")
print("Loaded!")
print(dataset)

df_original = dataset.to_pandas()

print(df_original.loc[1, "biased_text"])  # Note: .loc to access a specific row and column
print(df_original.head(5))


In [ ]:
# Design the dataset for binary classification: 1 for biased, 0 for neutral/unbiased
df_biased = pd.DataFrame({'text': df_original['biased_text'], 'label': 1})
df_unbiased = pd.DataFrame({'text': df_original['debiased_text'], 'label': 0})

# Shuffle to prevent skewed learning w/ .concat and .sample
df = pd.concat([df_biased, df_unbiased]).sample(frac=1, random_state=42).reset_index(drop=True)
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], 
    df['label'], 
    train_size=0.8,
    test_size=0.2)

print(f"Total sentences: {len(df)}")
print("Sample data:")
display(df.head(3))

In [ ]:
import re
from sklearn.feature_extraction.text import CountVectorizer
import contractions

def clean_text(text):
    if not isinstance(text, str):
        return ""
    for word, expanded in contractions.CONTRACTION_MAP.items():
        text = re.sub(r'\b' + re.escape(word) + r'\b', expanded, text, flags=re.IGNORECASE)
    return text.lower()

# CountVectorizer for Unigrams and Bigrams
vectorizer = CountVectorizer(
    preprocessor=clean_text, 
    ngram_range=(1, 2),
    max_features=10000)

print("Raw text to vector:")
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Success! Extracted {X_train_vec.shape[1]} unique unigram and bigram features.")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, log_loss

# Minimize Cross-Entropy loss via gradient descent
print("Training the Logistic Regression model...")
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)
print("Training is done!")

y_pred = model.predict(X_test_vec)             
y_pred_proba = model.predict_proba(X_test_vec) 
##########################################################################
print("Model performance:")
print(f"Accuracy:                   {accuracy_score(y_test, y_pred):.4f}")
print(f"Cross-Entropy Loss:         {log_loss(y_test, y_pred_proba):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))


In [ ]:
import numpy as np

feature_names = np.array(vectorizer.get_feature_names_out())
weights = model.coef_[0]
weight_df = pd.DataFrame({'Feature': feature_names, 'Weight': weights})

# Top BIASED indicators (highest positive weights)
print("Top words that indicate BIASED text (Class 1):")
display(weight_df.sort_values(by='Weight', ascending=False).head(10))

# Top NEUTRAL indicators (Highest negative weights)
print("Top words that indicate NEUTRAL text (Class 0):")
display(weight_df.sort_values(by='Weight', ascending=True).head(10))

In [ ]:
# Test the model on some new sentences
test_sentence =  "The public fucking hates the tariffs"

sentence_vec = vectorizer.transform([test_sentence])

prediction = model.predict(sentence_vec)[0]
# print(f"The text is (0 = Neutral, 1 = Biased): {prediction}")

# Confidence for each class (0=Neutral, 1=Biased)
class_confidence = model.predict_proba(sentence_vec).flatten()

# Print the final verdict
print(f"Input Text: '{test_sentence}'\n")

if prediction == 1:
    print(f"Verdict: 🚨 BIASED")
    print(f"Confidence: {class_confidence[1] * 100:.2f}%")
else:
    print(f"Verdict: ✅ NEUTRAL")
    print(f"Confidence: {class_confidence[0] * 100:.2f}%")

Future work:
- Try more complex models (Neural Networks, Transformers)
- Use more advanced features (TF-IDF, word embeddings)